In [ ]:
from collections import Counter, defaultdict
from typing import List, Dict, Literal, Union

import random
import re
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

import nltk
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from wordcloud import WordCloud, STOPWORDS
from nltk.stem.snowball import SnowballStemmer

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, TfidfTransformer
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

nltk.download('stopwords')
nltk.download('punkt')
sns.set_theme(style="whitegrid")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CACHE_DIR = "/storage/student6/.cache"
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxx"

# **1. Dataset**

## Load dataset from HuggingFace

In [ ]:
data = load_dataset("UniverseTBD/arxiv-abstracts-large", cache_dir=CACHE_DIR)
data

In [ ]:
# Print 5 random samples
for i in range(5):
    print(f"Sample {i+1}:")
    print(data['train'][i])
    print("-" * 50)

# **2. Data Analysis**

In [ ]:
all_categories = data['train']['categories']
print(set(all_categories))

In [ ]:
all_categories = data['train']['categories']
category_set = set()

# Collect unique labels
for category in all_categories:
    parts = category.split(' ')
    for part in parts:
        topic = part.split('.')[0]
        category_set.add(topic)

# Sort the labels and print them
sorted_categories= sorted(list(category_set), key=lambda x: x.lower())
print(f'There are {len(sorted_categories)} unique primary categories in the dataset:')
for category in sorted_categories:
    print(category)

## Label Grouping

In [ ]:
MAIN_CATS = [
    'astro-ph', 'cond-mat', 'cs', 'math', 'physics', 'q-bio',
    'nlin', 'hep-th', 'hep-ph', 'gr-qc', 'nucl-th', 'nucl-ex'
]
OTHER = "other"

MAP_TO_MAIN = {
    # physics-ish
    'acc-phys': 'physics',
    'atom-ph': 'physics',
    'chem-ph': 'physics',
    'plasm-ph': 'physics',
    'quant-ph': 'physics',

    # cond-mat-ish
    'mtrl-th': 'cond-mat',
    'supr-con': 'cond-mat',

    # math/stat-ish
    'alg-geom': 'math',
    'dg-ga': 'math',
    'funct-an': 'math',
    'math-ph': 'math',
    'q-alg': 'math',
    'stat': 'math',
    'solv-int': 'math',
    'bayes-an': 'math',

    # nonlinear-ish
    'chao-dyn': 'nlin',
    'patt-sol': 'nlin',
    'cmp-lg': 'nlin',
    'adap-org': 'nlin',

    # cs-ish
    'eess': 'cs',
    'q-fin': 'cs',

    # hep legacy
    'hep-ex': 'hep-ph',
    'hep-lat': 'hep-th',
}

DROP_TO_OTHER = {'ao-sci', 'comp-gas', 'econ'}


In [ ]:
def split_tokens(cat_str: str):
    return cat_str.strip().split() if isinstance(cat_str, str) else []

def base_label(token: str) -> str:
    return token.split('.')[0].strip()

def group_label(base: str) -> str:
    if base in MAIN_CATS:
        return base
    if base in DROP_TO_OTHER:
        return OTHER
    return MAP_TO_MAIN.get(base, OTHER)

def grouped_primary_label(cat_str: str) -> str:
    toks = split_tokens(cat_str)
    if not toks:
        return OTHER
    return group_label(base_label(toks[0]))

def grouped_all_labels(cat_str: str):
    toks = split_tokens(cat_str)
    if not toks:
        return [OTHER]
    return [group_label(base_label(t)) for t in toks]

## Select 10000 samples after grouping

In [ ]:
N_SAMPLES = 10000

train_shuf = data["train"].shuffle(seed=SEED)
samples = []

for s in train_shuf:
    toks = split_tokens(s["categories"])
    if not toks:
        continue

    lab = grouped_primary_label(s["categories"])  # grouped label

    samples.append({
        "abstract": s["abstract"],
        "orig_categories": s["categories"],
        "grouped_label": lab
    })

    if len(samples) >= N_SAMPLES:
        break

print(f"Number of samples: {len(samples)}")

for smp in random.sample(samples, k=min(3, len(samples))):
    print(f"Grouped label: {smp['grouped_label']} | Original: {smp['orig_categories']}")
    print("Abstract:", smp["abstract"][:600], "..." if len(smp["abstract"]) > 600 else "")
    print("#" * 20 + "\n")

## Label Distribution

In [ ]:
MAX_LABEL_DOCS = 200000
train_eda = data["train"].shuffle(seed=42).select(range(min(MAX_LABEL_DOCS, len(data["train"]))))

rows = []
for s in train_eda:
    toks = split_tokens(s["categories"])
    if not toks:
        continue
    rows.append({
        "n_labels": len(toks),
        "primary_grouped": grouped_primary_label(s["categories"]),
        "all_grouped": grouped_all_labels(s["categories"])
    })

df_labels = pd.DataFrame(rows)
df_single = df_labels[df_labels["n_labels"] == 1].copy()
df_multi  = df_labels[df_labels["n_labels"] >= 2].copy()

cat_order = MAIN_CATS + [OTHER]

In [ ]:
# 1) Primary grouped distribution (single-label only)
plt.figure(figsize=(10, 4))
sns.countplot(data=df_single, x="primary_grouped", order=cat_order)
plt.xticks(rotation=45, ha="right")
plt.title("Primary grouped label distribution (single-label only)")
plt.tight_layout()
plt.show()

In [ ]:
# 2) Primary grouped distribution (multi-label only)
plt.figure(figsize=(10, 4))
sns.countplot(data=df_multi, x="primary_grouped", order=cat_order)
plt.xticks(rotation=45, ha="right")
plt.title("Primary grouped label distribution (multi-label only)")
plt.tight_layout()
plt.show()

In [ ]:
# 3) All grouped occurrences (multi-label; token-count)
all_occ = df_multi["all_grouped"].explode()
df_all_occ = all_occ.to_frame(name="grouped")
plt.figure(figsize=(10, 4))
sns.countplot(data=df_all_occ, x="grouped", order=cat_order)
plt.xticks(rotation=45, ha="right")
plt.title("All grouped label occurrences (multi-label; token-count)")
plt.tight_layout()
plt.show()


In [ ]:
# 4) All grouped occurrences (multi-label; unique-per-abstract)
unique_per_abs = df_multi["all_grouped"].apply(lambda xs: sorted(set(xs))).explode()
df_unique = unique_per_abs.to_frame(name="grouped")
plt.figure(figsize=(10, 4))
sns.countplot(data=df_unique, x="grouped", order=cat_order)
plt.xticks(rotation=45, ha="right")
plt.title("All grouped labels (multi-label; unique-per-abstract)")
plt.tight_layout()
plt.show()

In [ ]:
# 5) Histogram: number of original labels per abstract
plt.figure(figsize=(6, 4))
sns.histplot(data=df_labels, x="n_labels", bins=range(1, df_labels["n_labels"].max() + 2), discrete=True)
plt.title("Histogram: number of original labels per abstract")
plt.xlabel("# original labels")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
print("\nGrouped primary label counts (all data):")
print(df_labels["primary_grouped"].value_counts().reindex(cat_order, fill_value=0))

# **3. Text Processing**

## Stopwords

In [ ]:
import unicodedata

KEEP_SHORT = {"ai", "ml", "dl", "rl", "nlp"}

stop_words = set(stopwords.words('english'))

# Extra stopwords: use for WORDCLOUD only
EXTRA_STOP = {
    "paper","result","results","method","methods","model","models","approach",
    "data","analysis","using","based","show","propose","study","studies",
    "problem","problems"
}

## Clean text

In [ ]:
def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\n", " ").replace("\t", " ").strip()
    text = re.sub(r"(http\S+|www\.\S+)", " ", text)   # URLs
    text = re.sub(r"\S+@\S+", " ", text)             # emails

    # Keep letters, digits, spaces, and hyphens
    text = re.sub(r"[^A-Za-z0-9\-\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

## Lemmatizing

In [ ]:
def preprocess_doc_lemma_only(doc) -> str:
    out = []
    for tok in doc:
        if tok.is_space:
            continue

        # drop standalone hyphen tokens if any
        if tok.text == "-":
            continue

        # keep numbers as a marker
        if tok.like_num:
            out.append("<num>")
            continue

        # keep alphabetic tokens (spaCy will split hyphenated words into pieces anyway)
        if not tok.is_alpha:
            continue

        lemma = tok.lemma_.lower().strip()

        # Fix spaCy quirk: data -> datum
        if lemma == "datum":
            lemma = "data"

        if not lemma:
            continue

        # stopwords: spaCy stoplist OR NLTK list
        if tok.is_stop or lemma in stop_words:
            continue

        # keep short acronyms
        if len(lemma) < 2 and lemma not in KEEP_SHORT:
            continue

        out.append(lemma)

    return " ".join(out)

## Word Cloud

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

def make_wordcloud(processed_texts, title, max_words=200):
    big = " ".join(processed_texts)

    wc = WordCloud(
        width=1200,
        height=600,
        background_color="white",
        max_words=max_words,
        stopwords=set(STOPWORDS).union(EXTRA_STOP),  # <-- EXTRA_STOP used here
        collocations=False,
        random_state=42
    ).generate(big)

    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

In [ ]:
MAX_DOCS = 30000
train = data["train"].shuffle(seed=42).select(range(min(MAX_DOCS, len(data["train"]))))

cleaned = []
labels = []
for s in train:
    cleaned.append(clean_text(s["abstract"]))
    labels.append(grouped_primary_label(s["categories"]))

# Lemmatize in batches (lemma-only)
processed = []
for doc in nlp.pipe(cleaned, batch_size=64):
    processed.append(preprocess_doc_lemma_only(doc))

In [ ]:
# Overall wordcloud (after full preprocessing)
make_wordcloud(processed, title=f"WordCloud (after preprocessing, N={len(processed)})")

## Print some random samples of processed texts and labels

In [ ]:
# Build df from processed texts + labels
df = pd.DataFrame({
    "text": processed,
    "label": labels
})

# Drop empty texts
df = df[df["text"].astype(str).str.strip().str.len() > 0].reset_index(drop=True)

print("df shape:", df.shape)
print(df["label"].value_counts())

In [ ]:
k = 5
idxs = random.sample(range(len(df)), k=min(k, len(df)))

for i in idxs:
    print(f"Label: {df.loc[i, 'label']}")
    txt = df.loc[i, "text"]
    print(f"Preprocessed text: {txt[:600]}{'...' if len(txt) > 600 else ''}")
    print("#" * 25 + "\n")

# **4. Map categorical labels to numerical IDs**

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

id2label = {i: lab for i, lab in enumerate(le.classes_)}
label2id = {lab: i for i, lab in id2label.items()}

print("Number of classes:", len(le.classes_))
print("label2id:", label2id)

In [ ]:
X_text = df["text"].tolist()
y_ref  = df["label_id"].values   # optional: only for plots / extrinsic comparison

# **5. Text Encoder**

## 5.1 Bag of Words

Bag of Words (BoW) is a simple and commonly used text representation technique. It converts text into a fixed-length vector by counting the occurrences of each word in the text. This method ignores grammar and word order but retains the frequency of words.

Bag of Words (BoW) là một kỹ thuật biểu diễn văn bản đơn giản và phổ biến. Nó chuyển đổi văn bản thành một vector có độ dài cố định bằng cách đếm số lần xuất hiện của mỗi từ trong văn bản. Phương pháp này bỏ qua ngữ pháp và trật tự từ nhưng vẫn giữ lại tần suất xuất hiện của từ.

In [ ]:
docs = [
    "I am going to school to study for the final exam.",
    "The weather is nice today and I feel happy.",
    "I love programming in Python and exploring new libraries.",
    "Data science is an exciting field with many opportunities.",
]

bow = CountVectorizer()
vectors = bow.fit_transform(docs)

for i, vec in enumerate(vectors):
    print(f"Document {i+1}: {vec.toarray()}")

In [ ]:
bow_vectorizer = CountVectorizer(
    max_features=200000,
    min_df=5,
    max_df=0.95
)
X_bow = bow_vectorizer.fit_transform(X_text)
print("Shape of X_bow:", X_bow.shape)

## 5.2 TF-IDF

Tf-idf (Term Frequency-Inverse Document Frequency) is another popular text representation technique. It not only considers the frequency of words in a document but also how common or rare a word is across all documents. This helps to reduce the weight of common words and highlight more informative ones.

TF-IDF (Term Frequency-Inverse Document Frequency) là một kỹ thuật biểu diễn văn bản phổ biến khác. Nó không chỉ xem xét tần suất xuất hiện của từ trong một tài liệu mà còn cả mức độ phổ biến hay hiếm gặp của từ đó trong tất cả các tài liệu. Điều này giúp giảm trọng số của các từ phổ biến và làm nổi bật những từ mang nhiều thông tin hơn.

In [ ]:
docs = [
    "I am going to school to study for the final exam.",
    "The weather is nice today and I feel happy.",
    "I love programming in Python and exploring new libraries.",
    "Data science is an exciting field with many opportunities.",
]

tfidf = TfidfVectorizer()
tfidf_vectors = tfidf.fit_transform(docs)

for i, vec in enumerate(tfidf_vectors):
    print(f"TF-IDF for Document {i+1}:")
    print(vec.toarray())

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=200000,
    min_df=5,
    max_df=0.95
)
X_tfidf = tfidf_vectorizer.fit_transform(X_text)
print("Shape of X_tfidf:", X_tfidf.shape)

## 5.3. Word2Vec

Word2Vec is a neural embedding method that learns a dense vector for each word so that words appearing in similar contexts end up with similar vectors (e.g., “galaxy” and “star” become close in vector space).

It trains on plain text by sliding a context window across sentences and solving a simple prediction task:

CBOW (Continuous Bag of Words): predict the center word from surrounding context words.

Skip-gram: predict context words from the center word.

After training, each word is represented by a fixed-length vector (e.g., 100 dimensions by default). For document/abstract classification, a common trick is to convert an abstract into a single vector by averaging its word vectors (or using a weighted average), then train a classifier on those document vectors.

Word2Vec là một phương pháp embeddings mạng nơron học một vectơ dày đặc cho mỗi từ sao cho các từ xuất hiện trong ngữ cảnh tương tự sẽ có các vectơ tương tự (ví dụ: “galaxy” và “star” trở nên gần nhau trong không gian vectơ). 

Nó được huấn luyện trên văn bản thuần túy bằng cách trượt cửa sổ ngữ cảnh trên các câu và giải quyết một nhiệm vụ dự đoán đơn giản: 

CBOW (Continuous Bag of Words): dự đoán từ trung tâm từ các từ ngữ cảnh xung quanh. 

Skip-gram: dự đoán các từ ngữ cảnh từ từ trung tâm. 

Sau khi huấn luyện, mỗi từ được biểu diễn bằng một vectơ có độ dài cố định (ví dụ: mặc định là 100 chiều). Đối với phân loại tài liệu/tóm tắt, một thủ thuật phổ biến là chuyển đổi một bản tóm tắt thành một vectơ duy nhất bằng cách lấy trung bình các vectơ từ của nó (hoặc sử dụng trung bình có trọng số), sau đó huấn luyện một bộ phân loại trên các vectơ tài liệu đó.

In [ ]:
# Tokenize (all documents)
all_tokens = [t.split() for t in X_text]   # X_text = df["text"].tolist()

w2v = Word2Vec(
    sentences=all_tokens,
    vector_size=200,
    window=8,
    min_count=10,
    sg=1,
    negative=10,
    sample=1e-4,
    epochs=5,
    workers=max(1, os.cpu_count()-1),
    seed=42
)

print("Word2Vec vocab size:", len(w2v.wv))
print("Word2Vec vector size:", w2v.vector_size)

In [ ]:
def doc_vector_mean(tokens, model, dim):
    vecs = []
    for w in tokens:
        if w in model.wv:
            vecs.append(model.wv[w])
    if not vecs:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

In [ ]:
dim = w2v.vector_size

# Document embeddings by mean of word vectors
X_w2v = np.vstack([doc_vector_mean(toks, w2v, dim) for toks in all_tokens])

print("W2V shape:", X_w2v.shape)

## 5.4 Word Embeddings:

Word embeddings are dense vector representations of words that capture semantic relationships between them. They are typically pre-trained on large corpora and can be used to represent words in a continuous vector space, allowing for better generalization and understanding of word meanings.

Word Embeddings là các biểu diễn vector dày đặc của từ, nắm bắt các mối quan hệ ngữ nghĩa giữa chúng. Chúng thường được huấn luyện trước trên các kho ngữ liệu lớn và có thể được sử dụng để biểu diễn từ trong không gian vector liên tục, cho phép khái quát hóa tốt hơn và hiểu rõ hơn ý nghĩa của từ.

In [ ]:
from typing import List, Literal
import numpy as np
from sentence_transformers import SentenceTransformer

class EmbeddingVectorizer:
    def __init__(
        self,
        model_name: str = 'intfloat/multilingual-e5-base',
        normalize: bool = True,
        max_seq_length: int = 256,   # important for long abstracts
        device: str | None = None    # "cpu" / "cuda" 
    ):
        self.model = SentenceTransformer(model_name, device=device)  # device
        self.normalize = normalize
        self.model.max_seq_length = max_seq_length                  

    def _format_inputs(
        self,
        texts: List[str],
        mode: Literal['query', 'passage', 'raw']                    
    ) -> List[str]:
        if mode not in {"query", "passage", "raw"}:                 
            raise ValueError("Mode must be either 'query', 'passage', or 'raw'")
        if mode == "raw":
            return [text.strip() for text in texts]                 
        return [f"{mode}: {text.strip()}" for text in texts]

    def transform(
        self,
        texts: List[str],
        mode: Literal['query', 'passage', 'raw'] = 'passage',       # default passage for docs
        batch_size: int = 64,                                        
        show_progress_bar: bool = True                              
    ):
        inputs = self._format_inputs(texts, mode)                   # always use formatter
        embeddings = self.model.encode(
            inputs,
            normalize_embeddings=self.normalize,
            batch_size=batch_size,
            show_progress_bar=show_progress_bar
        )
        return embeddings.tolist()

    def transform_numpy(
        self,
        texts,
        mode: Literal['query', 'passage', 'raw'] = 'passage',       
        batch_size: int = 64,                                       
        show_progress_bar: bool = True                              
    ) -> np.ndarray:
        inputs = self._format_inputs(list(texts), mode)             
        embeddings = self.model.encode(
            inputs,
            normalize_embeddings=self.normalize,
            batch_size=batch_size,
            show_progress_bar=show_progress_bar
        )
        return np.asarray(embeddings, dtype=np.float32)             # ensure numpy float32

In [ ]:
vec = EmbeddingVectorizer(
    model_name="intfloat/multilingual-e5-base",
    normalize=True,
    max_seq_length=256,
    device="cpu"   # change to "cuda" only if available
)

X_st = vec.transform_numpy(X_text, mode="passage", batch_size=128, show_progress_bar=True)
print("E5 shape:", X_st.shape)

## Convert all embeddings to NumPy for Consistency

In [ ]:
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

def to_numpy(X):
    """
    Convert common feature formats to NumPy arrays for consistency:
    - NumPy stays NumPy
    - scipy sparse -> dense NumPy
    - Python lists -> NumPy
    """
    if isinstance(X, np.ndarray):
        return X
    if sparse.issparse(X):
        return X.toarray()  # WARNING: can be huge for BoW/TF-IDF
    return np.asarray(X)

# Keep sparse as-is (do NOT densify)
X_bow_sp   = X_bow
X_tfidf_sp = X_tfidf

# Reduce sparse -> dense for consistent clustering/viz
SVD_DIM = 50
bow_svd   = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
tfidf_svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)

X_bow_50   = bow_svd.fit_transform(X_bow_sp)
X_tfidf_50 = tfidf_svd.fit_transform(X_tfidf_sp)

# Dense embeddings already
X_w2v_np = to_numpy(X_w2v)
X_st_np  = to_numpy(X_st)

print("BoW sparse:",   X_bow_sp.shape,   "-> BoW_50:",   X_bow_50.shape)
print("TFIDF sparse:", X_tfidf_sp.shape, "-> TFIDF_50:", X_tfidf_50.shape)
print("W2V dense:",    X_w2v_np.shape)
print("E5 dense:",     X_st_np.shape)

# **6. Dimensionality Reduction and Visualization**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

In [ ]:
def sample_rows(X, y=None, n=5000, seed=42):
    rng = np.random.RandomState(seed)
    N = X.shape[0]
    n = min(n, N)
    idx = rng.choice(N, size=n, replace=False)

    Xs = X[idx, :]
    if y is None:
        return Xs, None
    ys = np.asarray(y)[idx]
    return Xs, ys

def plot_2d(Z, y, title):
    plt.figure(figsize=(8, 6))
    if y is None:
        plt.scatter(Z[:, 0], Z[:, 1], s=10, alpha=0.6)
    else:
        sc = plt.scatter(Z[:, 0], Z[:, 1], c=y, s=10, alpha=0.6)
        cb = plt.colorbar(sc)
        cb.set_label("label_id (reference only)")
    plt.title(title)
    plt.xlabel("dim-1")
    plt.ylabel("dim-2")
    plt.tight_layout()
    plt.show()

def run_tsne(X50, seed=42, perplexity=30):
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate="auto",
        init="pca",
        random_state=seed
    )
    return tsne.fit_transform(X50)

def run_umap(X50, seed=42, n_neighbors=15, min_dist=0.1, metric="euclidean"):
    if not HAS_UMAP:
        raise ImportError("UMAP not installed. Try: pip install umap-learn")
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric=metric,
        random_state=seed
    )
    return reducer.fit_transform(X50)

In [ ]:
embeddings_for_viz = {
    "BoW (SVD50)":   X_bow_50,
    "TFIDF (SVD50)": X_tfidf_50,
    "W2V (200d)":    X_w2v_np,
    "E5 (768d)":     X_st_np,
}

In [ ]:
# Color is optional: use y_ref if you want to see alignment with known categories
# If you don't have it, set y_color = None
y_color = y_ref  # or None

N_VIS = 6000

# ---- Quick 2D overview with PCA(2) ----
for name, X in embeddings_for_viz.items():
    Xs, ys = sample_rows(X, y=y_color, n=N_VIS, seed=SEED)
    Z = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
    plot_2d(Z, ys, f"{name} — PCA(2) (n={Xs.shape[0]})")

In [ ]:
# ---- Optional: TSNE/UMAP on a 50D space (recommended to use *_50 spaces) ----
# Example: TSNE on TFIDF_50
Xs, ys = sample_rows(X_tfidf_50, y=y_color, n=4000, seed=SEED)
Z_tsne = run_tsne(Xs, seed=SEED, perplexity=30)
plot_2d(Z_tsne, ys, f"TFIDF(SVD50) — t-SNE(2) (n={Xs.shape[0]})")

# Example: UMAP on E5 after first reducing to 50D with PCA (optional)
# This avoids running UMAP directly on 768D for speed.
Xs, ys = sample_rows(X_st_np, y=y_color, n=8000, seed=SEED)
X50 = PCA(n_components=50, random_state=SEED).fit_transform(Xs)
if HAS_UMAP:
    Z_umap = run_umap(X50, seed=SEED, n_neighbors=15, min_dist=0.1, metric="euclidean")
    plot_2d(Z_umap, ys, f"E5(PCA50) — UMAP(2) (n={Xs.shape[0]})")

# **7. Clustering Algorithms**

In [ ]:
import time
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans, DBSCAN, MeanShift, estimate_bandwidth, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

In [ ]:
def pca50_normalized(X, n_components=50, seed=42):
    """PCA -> 50D then normalize (no StandardScaler)."""
    if X.shape[1] <= n_components:
        return normalize(X)
    pca = PCA(n_components=n_components, random_state=seed, svd_solver="randomized")
    X50 = pca.fit_transform(X)
    return normalize(X50)

EMBEDDINGS = {
    "BoW_SVD50": {
        "X_k":  normalize(X_bow_50),   # for k-based models
        "X_50": normalize(X_bow_50),   # for DBSCAN/MeanShift
    },
    "TFIDF_SVD50": {
        "X_k":  normalize(X_tfidf_50),
        "X_50": normalize(X_tfidf_50),
    },
    "W2V_200": {
        "X_k":  normalize(X_w2v_np),
        "X_50": pca50_normalized(X_w2v_np, 50, SEED),
    },
    "E5_768": {
        "X_k":  normalize(X_st_np),
        "X_50": pca50_normalized(X_st_np, 50, SEED),
    },
}

print({k: (v["X_k"].shape, v["X_50"].shape) for k, v in EMBEDDINGS.items()})

## Helpers

In [ ]:
def sample_rows(X, y=None, n=8000, seed=42):
    rng = np.random.RandomState(seed)
    N = X.shape[0]
    n = min(n, N)
    idx = rng.choice(N, size=n, replace=False)
    Xs = X[idx, :]
    if y is None:
        return Xs, None
    return Xs, np.asarray(y)[idx]

def _count_clusters(labels):
    labels = np.asarray(labels)
    uniq = set(labels.tolist())
    return len(uniq) - (1 if -1 in uniq else 0)

def evaluate_clustering(X, labels, y_ref=None, sil_metric="euclidean", drop_noise_for_db_ch=True):
    labels = np.asarray(labels)
    out = {
        "n_clusters": _count_clusters(labels),
        "noise_frac": float(np.mean(labels == -1)) if np.any(labels == -1) else 0.0
    }

    # silhouette needs >=2 non-noise clusters
    uniq = set(labels.tolist())
    uniq_no_noise = [u for u in uniq if u != -1]
    if len(uniq_no_noise) >= 2:
        try:
            out["silhouette"] = float(silhouette_score(X, labels, metric=sil_metric))
        except Exception:
            out["silhouette"] = np.nan
    else:
        out["silhouette"] = np.nan

    # DBI/CH: optionally remove noise
    if drop_noise_for_db_ch and np.any(labels == -1):
        mask = labels != -1
        X2 = X[mask]
        lab2 = labels[mask]
    else:
        X2 = X
        lab2 = labels

    if _count_clusters(lab2) >= 2:
        try:
            out["dbi"] = float(davies_bouldin_score(X2, lab2))
        except Exception:
            out["dbi"] = np.nan
        try:
            out["ch"] = float(calinski_harabasz_score(X2, lab2))
        except Exception:
            out["ch"] = np.nan
    else:
        out["dbi"] = np.nan
        out["ch"] = np.nan

    if y_ref is not None:
        y_ref = np.asarray(y_ref)
        out["nmi"] = float(normalized_mutual_info_score(y_ref, labels))
        out["ari"] = float(adjusted_rand_score(y_ref, labels))
    else:
        out["nmi"] = np.nan
        out["ari"] = np.nan

    return out

def choose_best_k(df):
    # maximize silhouette, then CH, then minimize DBI
    d = df.copy()
    d["sil_f"] = d["silhouette"].fillna(-1e9)
    d["ch_f"]  = d["ch"].fillna(-1e9)
    d["db_f"]  = d["dbi"].fillna(1e9)
    return d.sort_values(["sil_f","ch_f","db_f"], ascending=[False, False, True]).iloc[0]

def plot_k_curves(df, title_prefix, include_inertia=False, include_aic_bic=False):
    d = df.sort_values("k").copy()
    ks = d["k"].values
    best = choose_best_k(d)
    best_k = int(best["k"])

    def _plot(col, ylabel, better="high"):
        if col not in d.columns:
            return
        plt.figure()
        plt.plot(ks, d[col].values, marker="o")
        plt.axvline(best_k, linestyle="--")
        plt.title(f"{title_prefix} — {ylabel} vs k (best k={best_k})")
        plt.xlabel("k")
        plt.ylabel(ylabel + (" (higher better)" if better=="high" else " (lower better)"))
        plt.grid(True)
        plt.show()

    _plot("silhouette", "Silhouette", "high")
    _plot("ch", "Calinski–Harabasz (CH)", "high")
    _plot("dbi", "Davies–Bouldin (DBI)", "low")

    if include_inertia:
        _plot("inertia", "Inertia (Elbow)", "low")
    if include_aic_bic:
        _plot("aic", "AIC", "low")
        _plot("bic", "BIC", "low")

    return best_k, best

In [ ]:
K_LIST = list(range(10, 61, 5))
EPS_LIST = [0.3, 0.5, 0.7, 0.9, 1.1]
MIN_SAMPLES_LIST = [5, 10, 20]
MS_QUANTILES = [0.10, 0.15, 0.20, 0.25]

N_AHC = 8000     # AHC subset size
N_MS  = 5000     # MeanShift subset size

def scan_kmeans(X, k_list, y_ref=None):
    rows = []
    for k in k_list:
        t0 = time.perf_counter()
        model = KMeans(n_clusters=k, n_init="auto", random_state=SEED)
        labels = model.fit_predict(X)
        fit_s = time.perf_counter() - t0
        m = evaluate_clustering(X, labels, y_ref=y_ref, sil_metric="euclidean")
        rows.append({"algo":"KMeans","k":k,"fit_s":fit_s,"inertia":float(model.inertia_),**m})
    return pd.DataFrame(rows)

def _make_ahc(k, linkage="ward", metric="euclidean"):
    try:
        return AgglomerativeClustering(n_clusters=k, linkage=linkage, metric=metric)
    except TypeError:
        return AgglomerativeClustering(n_clusters=k, linkage=linkage, affinity=metric)

def scan_ahc(X, k_list, y_ref=None, linkage="ward", metric="euclidean"):
    rows = []
    for k in k_list:
        t0 = time.perf_counter()
        model = _make_ahc(k, linkage=linkage, metric=metric)
        labels = model.fit_predict(X)
        fit_s = time.perf_counter() - t0
        m = evaluate_clustering(X, labels, y_ref=y_ref, sil_metric=metric)
        rows.append({"algo":f"AHC({linkage})","k":k,"fit_s":fit_s,**m})
    return pd.DataFrame(rows)

def scan_gmm(X, k_list, y_ref=None, covariance_type="diag"):
    rows = []
    for k in k_list:
        t0 = time.perf_counter()
        model = GaussianMixture(
            n_components=k,
            covariance_type=covariance_type,
            random_state=SEED,
            reg_covar=1e-6
        )
        model.fit(X)
        labels = model.predict(X)
        fit_s = time.perf_counter() - t0
        m = evaluate_clustering(X, labels, y_ref=y_ref, sil_metric="euclidean")
        rows.append({"algo":f"GMM({covariance_type})","k":k,"fit_s":fit_s,
                     "aic":float(model.aic(X)),"bic":float(model.bic(X)),**m})
    return pd.DataFrame(rows)

def scan_dbscan(X, eps_list, min_samples_list, y_ref=None, metric="euclidean"):
    rows = []
    for eps in eps_list:
        for ms in min_samples_list:
            t0 = time.perf_counter()
            model = DBSCAN(eps=eps, min_samples=ms, metric=metric)
            labels = model.fit_predict(X)
            fit_s = time.perf_counter() - t0
            m = evaluate_clustering(X, labels, y_ref=y_ref, sil_metric=metric)
            rows.append({"algo":"DBSCAN","eps":eps,"min_samples":ms,"fit_s":fit_s,**m})
    return pd.DataFrame(rows)

def pick_best_dbscan(df):
    # need >=2 clusters; maximize silhouette then minimize noise
    d = df[df["n_clusters"] >= 2].copy()
    if len(d) == 0:
        return None
    d["sil_f"] = d["silhouette"].fillna(-1e9)
    d["noise_f"] = d["noise_frac"].fillna(1e9)
    return d.sort_values(["sil_f","noise_f"], ascending=[False, True]).iloc[0]

def scan_meanshift(X, quantiles, y_ref=None, max_n=5000):
    Xs = X[:max_n]
    yS = y_ref[:max_n] if y_ref is not None else None
    rows = []
    for q in quantiles:
        t0 = time.perf_counter()
        bw = estimate_bandwidth(Xs, quantile=q, n_samples=min(2000, len(Xs)))
        model = MeanShift(bandwidth=bw, bin_seeding=True)
        labels = model.fit_predict(Xs)
        fit_s = time.perf_counter() - t0
        m = evaluate_clustering(Xs, labels, y_ref=yS, sil_metric="euclidean")
        rows.append({"algo":"MeanShift","quantile":q,"bandwidth":bw,"fit_s":fit_s,**m})
    return pd.DataFrame(rows)

def pick_best_meanshift(df):
    # maximize silhouette, then CH, then minimize DBI
    d = df.copy()
    d["k"] = (d["quantile"]*100).astype(int)  # dummy for choose_best_k
    return choose_best_k(d)

## Run the clustering

In [ ]:
RESULTS = []          # final summary rows
DETAILS = {}          # store scan tables per embedding

y_use = y_ref if "y_ref" in globals() else None

for emb_name, spaces in EMBEDDINGS.items():
    print("\n" + "="*80)
    print("Embedding:", emb_name)

    X_k  = spaces["X_k"]    # for KMeans
    X_50 = spaces["X_50"]   # for AHC/GMM/DBSCAN/MeanShift

    DETAILS[emb_name] = {}

    # --------------------
    # KMeans (scan k + curves)
    # --------------------
    df_km = scan_kmeans(X_k, K_LIST, y_ref=y_use)
    DETAILS[emb_name]["KMeans_scan"] = df_km
    best_k_km, best_row_km = plot_k_curves(df_km, f"{emb_name} — KMeans", include_inertia=True)
    kmeans_final = KMeans(n_clusters=int(best_k_km), n_init="auto", random_state=SEED)
    labels_km = kmeans_final.fit_predict(X_k)
    m_km = evaluate_clustering(X_k, labels_km, y_ref=y_use)
    RESULTS.append({"embedding":emb_name,"algo":"KMeans","chosen_k":int(best_k_km),**m_km})

    # --------------------
    # AHC (scan k + curves)
    # --------------------
    X_ahc, y_ahc = sample_rows(X_50, y=y_use, n=N_AHC, seed=SEED)
    df_ahc = scan_ahc(X_ahc, K_LIST, y_ref=y_ahc, linkage="ward", metric="euclidean")
    DETAILS[emb_name]["AHC_scan"] = df_ahc
    best_k_ahc, best_row_ahc = plot_k_curves(df_ahc, f"{emb_name} — AHC (subset n={X_ahc.shape[0]})")
    ahc_final = _make_ahc(int(best_k_ahc), linkage="ward", metric="euclidean")
    labels_ahc = ahc_final.fit_predict(X_ahc)
    m_ahc = evaluate_clustering(X_ahc, labels_ahc, y_ref=y_ahc)
    RESULTS.append({"embedding":emb_name,"algo":"AHC(subset)","chosen_k":int(best_k_ahc),**m_ahc})

    # --------------------
    # EM (scan k + curves + AIC/BIC)
    # --------------------
    X_gmm, y_gmm = sample_rows(X_50, y=y_use, n=N_AHC, seed=SEED)
    df_gmm = scan_gmm(X_gmm, K_LIST, y_ref=y_gmm, covariance_type="diag")
    DETAILS[emb_name]["GMM_scan"] = df_gmm
    best_k_gmm, best_row_gmm = plot_k_curves(df_gmm, f"{emb_name} — EM/GMM (subset n={X_gmm.shape[0]})", include_aic_bic=True)
    gmm_final = GaussianMixture(n_components=int(best_k_gmm), covariance_type="diag", random_state=SEED, reg_covar=1e-6)
    gmm_final.fit(X_gmm)
    labels_gmm = gmm_final.predict(X_gmm)
    m_gmm = evaluate_clustering(X_gmm, labels_gmm, y_ref=y_gmm)
    RESULTS.append({"embedding":emb_name,"algo":"EM/GMM(subset)","chosen_k":int(best_k_gmm),**m_gmm})

    # --------------------
    # DBSCAN (full on X_50; no k)
    # --------------------
    df_db = scan_dbscan(X_50, EPS_LIST, MIN_SAMPLES_LIST, y_ref=y_use, metric="euclidean")
    DETAILS[emb_name]["DBSCAN_scan"] = df_db
    best_db = pick_best_dbscan(df_db)
    if best_db is None:
        print(f"{emb_name} — DBSCAN: no setting produced >=2 clusters. Try adjusting EPS_LIST.")
        RESULTS.append({"embedding":emb_name,"algo":"DBSCAN","chosen_k":np.nan,"n_clusters":0,"noise_frac":1.0,
                        "silhouette":np.nan,"dbi":np.nan,"ch":np.nan,"nmi":np.nan,"ari":np.nan})
    else:
        dbscan_final = DBSCAN(eps=float(best_db["eps"]), min_samples=int(best_db["min_samples"]), metric="euclidean")
        labels_db = dbscan_final.fit_predict(X_50)
        m_db = evaluate_clustering(X_50, labels_db, y_ref=y_use, sil_metric="euclidean")
        RESULTS.append({"embedding":emb_name,"algo":"DBSCAN","chosen_k":np.nan,"eps":float(best_db["eps"]),
                        "min_samples":int(best_db["min_samples"]),**m_db})

    # --------------------
    # MeanShift (subset; no k)
    # --------------------
    X_ms, y_ms = sample_rows(X_50, y=y_use, n=N_MS, seed=SEED)
    df_ms = scan_meanshift(X_ms, MS_QUANTILES, y_ref=y_ms, max_n=X_ms.shape[0])
    DETAILS[emb_name]["MeanShift_scan"] = df_ms
    best_ms = pick_best_meanshift(df_ms)
    meanshift_final = MeanShift(bandwidth=float(best_ms["bandwidth"]), bin_seeding=True)
    labels_ms = meanshift_final.fit_predict(X_ms)
    m_ms = evaluate_clustering(X_ms, labels_ms, y_ref=y_ms)
    RESULTS.append({"embedding":emb_name,"algo":"MeanShift(subset)","chosen_k":np.nan,
                    "quantile":float(best_ms["quantile"]), "bandwidth":float(best_ms["bandwidth"]), **m_ms})

# Final comparison table
summary = pd.DataFrame(RESULTS)
display(summary.sort_values(["embedding","algo"]))

# **8. Visualize Clusters**

In [ ]:
def project_2d(X, method="pca", seed=42):
    """
    Project X to 2D for visualization.
    - method: 'pca' (fast), 'tsne' (slow), 'umap' (fast if installed)
    - If X has many dims, reduce to 50D with PCA first for tsne/umap stability.
    """
    X = np.asarray(X)
    if X.shape[1] > 50 and method in ["tsne", "umap"]:
        X50 = PCA(n_components=50, random_state=seed).fit_transform(X)
    else:
        X50 = X

    if method == "pca":
        return PCA(n_components=2, random_state=seed).fit_transform(X)
    if method == "tsne":
        return TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=seed).fit_transform(X50)
    if method == "umap":
        if not HAS_UMAP:
            raise ImportError("UMAP not installed. pip install umap-learn")
        return umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=seed).fit_transform(X50)
    raise ValueError("method must be one of: pca, tsne, umap")

def plot_clusters_2d_with_legend(
    Z, labels, title,
    max_legend_items=15,     # top-N clusters in legend
    noise_label=-1,
    point_size=10,
    alpha=0.75
):
    """
    Scatter plot with a readable legend:
    - Shows legend entries for top-N largest clusters.
    - Others grouped into "Other".
    - Noise label (-1) shown separately if present.
    """
    Z = np.asarray(Z)
    labels = np.asarray(labels)

    # Count cluster sizes
    uniq, counts = np.unique(labels, return_counts=True)
    size_map = dict(zip(uniq.tolist(), counts.tolist()))

    # Separate noise
    has_noise = noise_label in size_map
    if has_noise:
        noise_count = size_map[noise_label]

    # Sort clusters by size (exclude noise)
    clusters = [(c, size_map[c]) for c in size_map.keys() if c != noise_label]
    clusters.sort(key=lambda x: x[1], reverse=True)

    # Choose top-N clusters for legend
    top_clusters = [c for c, _ in clusters[:max_legend_items]]
    top_set = set(top_clusters)

    # Build a plotting label per point:
    # - Top clusters keep their own id
    # - Non-top non-noise -> "Other"
    # - Noise -> "Noise"
    plot_group = labels.astype(object)
    for i, lab in enumerate(labels):
        if lab == noise_label and has_noise:
            plot_group[i] = "Noise"
        elif lab in top_set:
            plot_group[i] = f"Cluster {lab}"
        else:
            plot_group[i] = "Other"

    # Unique groups in desired order
    groups = []
    for c in top_clusters:
        groups.append(f"Cluster {c}")
    if "Other" in plot_group:
        groups.append("Other")
    if has_noise:
        groups.append("Noise")

    plt.figure(figsize=(8, 6))

    # Plot each group separately so legend works
    for g in groups:
        mask = (plot_group == g)
        plt.scatter(Z[mask, 0], Z[mask, 1], s=point_size, alpha=alpha, label=g)

    plt.title(title)
    plt.xlabel("dim-1")
    plt.ylabel("dim-2")
    plt.tight_layout()

    # Put legend outside if it’s long
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True)
    plt.show()

In [ ]:
# For AHC/GMM/MeanShift we re-fit on a reproducible subset to keep runtime reasonable.
N_AHC = 8000
N_MS  = 5000

def sample_idx(n_total, n_sample, seed=42):
    rng = np.random.RandomState(seed)
    n = min(n_sample, n_total)
    return rng.choice(n_total, size=n, replace=False)

def _make_ahc(k, linkage="ward", metric="euclidean"):
    try:
        return AgglomerativeClustering(n_clusters=k, linkage=linkage, metric=metric)
    except TypeError:
        return AgglomerativeClustering(n_clusters=k, linkage=linkage, affinity=metric)

FINAL = {}  # FINAL[emb][algo] = dict(model=..., labels=..., space='X_k'/'X_50', idx=...)

for emb_name, spaces in EMBEDDINGS.items():
    print("\n" + "="*80)
    print("Embedding:", emb_name)
    FINAL[emb_name] = {}

    X_k  = spaces["X_k"]   # used for KMeans
    X_50 = spaces["X_50"]  # used for AHC/GMM/DBSCAN/MeanShift
    y_use = y_ref if "y_ref" in globals() else None

    # ---------- KMeans (full) ----------
    df_km = DETAILS[emb_name]["KMeans_scan"]
    best_k_km = int(choose_best_k(df_km)["k"])
    km = KMeans(n_clusters=best_k_km, n_init="auto", random_state=SEED)
    labels_km = km.fit_predict(X_k)
    FINAL[emb_name]["KMeans"] = {"model": km, "labels": labels_km, "space": "X_k", "idx": None}

    Z_km = project_2d(X_k, method="pca", seed=SEED)
    plot_clusters_2d_with_legend(Z_km, labels_km, f"{emb_name} — KMeans (k={best_k_km})")

    # ---------- AHC (subset) ----------
    idx_ahc = sample_idx(X_50.shape[0], N_AHC, seed=SEED)
    X_ahc = X_50[idx_ahc]
    y_ahc = y_use[idx_ahc] if y_use is not None else None

    df_ahc = DETAILS[emb_name]["AHC_scan"]
    best_k_ahc = int(choose_best_k(df_ahc)["k"])
    ahc = _make_ahc(best_k_ahc, linkage="ward", metric="euclidean")
    labels_ahc = ahc.fit_predict(X_ahc)
    FINAL[emb_name]["AHC"] = {"model": ahc, "labels": labels_ahc, "space": "X_50", "idx": idx_ahc}

    Z_ahc = project_2d(X_ahc, method="pca", seed=SEED)
    plot_clusters_2d_with_legend(Z_ahc, labels_ahc, f"{emb_name} — AHC (subset n={len(idx_ahc)}, k={best_k_ahc})")

    # ---------- EM / GMM (subset) ----------
    idx_gmm = idx_ahc  # reuse same subset for fairness
    X_gmm = X_50[idx_gmm]
    y_gmm = y_use[idx_gmm] if y_use is not None else None

    df_gmm = DETAILS[emb_name]["GMM_scan"]
    best_k_gmm = int(choose_best_k(df_gmm)["k"])
    gmm = GaussianMixture(n_components=best_k_gmm, covariance_type="diag", random_state=SEED, reg_covar=1e-6)
    gmm.fit(X_gmm)
    labels_gmm = gmm.predict(X_gmm)
    FINAL[emb_name]["GMM"] = {"model": gmm, "labels": labels_gmm, "space": "X_50", "idx": idx_gmm}

    Z_gmm = project_2d(X_gmm, method="pca", seed=SEED)
    plot_clusters_2d_with_legend(Z_gmm, labels_gmm, f"{emb_name} — EM/GMM (subset n={len(idx_gmm)}, k={best_k_gmm})")

    # ---------- DBSCAN (full) ----------
    df_db = DETAILS[emb_name]["DBSCAN_scan"]
    best_db = pick_best_dbscan(df_db)
    if best_db is not None:
        eps = float(best_db["eps"])
        ms = int(best_db["min_samples"])
        db = DBSCAN(eps=eps, min_samples=ms, metric="euclidean")
        labels_db = db.fit_predict(X_50)
        FINAL[emb_name]["DBSCAN"] = {"model": db, "labels": labels_db, "space": "X_50", "idx": None}

        Z_db = project_2d(X_50, method="pca", seed=SEED)
        plot_clusters_2d_with_legend(Z_db, labels_db, f"{emb_name} — DBSCAN (eps={eps}, min_samples={ms})")
    else:
        print(f"{emb_name} — DBSCAN: no params produced >=2 clusters; skip plot.")

    # ---------- MeanShift (subset) ----------
    idx_ms = sample_idx(X_50.shape[0], N_MS, seed=SEED)
    X_ms = X_50[idx_ms]
    y_ms = y_use[idx_ms] if y_use is not None else None

    df_ms = DETAILS[emb_name]["MeanShift_scan"]
    best_ms = pick_best_meanshift(df_ms)
    bw = float(best_ms["bandwidth"])
    ms_model = MeanShift(bandwidth=bw, bin_seeding=True)
    labels_ms = ms_model.fit_predict(X_ms)
    FINAL[emb_name]["MeanShift"] = {"model": ms_model, "labels": labels_ms, "space": "X_50", "idx": idx_ms}

    Z_ms = project_2d(X_ms, method="pca", seed=SEED)
    plot_clusters_2d_with_legend(Z_ms, labels_ms, f"{emb_name} — MeanShift (subset n={len(idx_ms)}, bw={bw:.3g})")

print("\nVisualization done. Models + labels stored in FINAL dict.")

In [ ]:
import json

# 1. Chọn tổ hợp Embedding và Model mà bạn muốn dùng làm chuẩn trên Web (Ví dụ: TFIDF + KMeans)
target_emb = "TFIDF_SVD50"
target_algo = "KMeans"

# 2. Lấy nhãn dự đoán từ biến FINAL và nhãn gốc từ dataframe df
# 'labels_km' ở đây lấy từ kết quả đã lưu trong FINAL
labels_to_map = FINAL[target_emb][target_algo]["labels"]

# 3. Tạo DataFrame để đối soát
# Lưu ý: df['label'] chính là cột chứa 'cs', 'math', 'physics'... bạn đã tạo ở phần Preprocessing
mapping_df = pd.DataFrame({
    'cluster_id': labels_to_map,
    'ground_truth': df['label'] 
})

# 4. Tự động tìm nhãn ngành học chiếm đa số trong mỗi Cluster
cluster_to_topic = {}
print(f"--- Mapping results for {target_emb} with {target_algo} ---")

unique_clusters = sorted([c for c in np.unique(labels_to_map) if c != -1])

for c_id in unique_clusters:
    # Lấy danh sách các nhãn gốc nằm trong Cluster này
    subset = mapping_df[mapping_df['cluster_id'] == c_id]
    if not subset.empty:
        # Tìm nhãn xuất hiện nhiều nhất (Majority Vote)
        majority_label = subset['ground_truth'].value_counts().idxmax()
        cluster_to_topic[int(c_id)] = majority_label
        print(f"Cluster {c_id} is mainly: {majority_label}")

# 5. Lưu file mapping này vào thư mục artifacts để app.py sử dụng
mapping_path = os.path.join(ART_DIR, "cluster_mapping.json")
with open(mapping_path, "w") as f:
    json.dump(cluster_to_topic, f)

print(f"\nSuccessfully saved mapping to: {mapping_path}")

# **9. Save artifacts**

In [ ]:
import os, json
import joblib
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA

ART_DIR = "artifacts_clustering"
os.makedirs(ART_DIR, exist_ok=True)

normalizer = Normalizer(norm="l2")  # "scaler" equivalent for normalize-only pipelines

# Optional: create PCA50 objects for W2V/E5 if you want to reproduce X_50 in deployment
# (BoW/TFIDF already have SVD objects: bow_svd/tfidf_svd)
pca_w2v_50 = None
pca_e5_50  = None
if "X_w2v_np" in globals():
    pca_w2v_50 = PCA(n_components=50, random_state=SEED).fit(X_w2v_np)
if "X_st_np" in globals():
    pca_e5_50  = PCA(n_components=50, random_state=SEED).fit(X_st_np)

def safe_feature_names(vec, limit=200000):
    try:
        names = vec.get_feature_names_out()
        return names[:min(len(names), limit)].tolist()
    except Exception:
        return None

VOCAB_LIMIT = 200000

saved_index = []  # records what we saved

for emb_name, algos in FINAL.items():
    for algo_name, pack in algos.items():
        model = pack["model"]

        artifact = {
            "embedding_name": emb_name,
            "algorithm": algo_name,
            "normalizer": normalizer,        # normalize-only "scaler"
            "model": model,
            "meta": {
                "space_used": pack["space"],
                "subset_idx": pack["idx"].tolist() if pack["idx"] is not None else None,
                "seed": SEED,
            },
            "preprocess": {},
            "vocab_or_tokens": {}
        }

        # Attach preprocess + vocab depending on embedding
        if emb_name.startswith("BoW"):
            if "bow_vectorizer" in globals(): artifact["preprocess"]["vectorizer"] = bow_vectorizer
            if "bow_svd" in globals():        artifact["preprocess"]["svd_50"] = bow_svd
            if "bow_vectorizer" in globals():
                artifact["vocab_or_tokens"]["tokens"] = safe_feature_names(bow_vectorizer, limit=VOCAB_LIMIT)

        elif emb_name.startswith("TFIDF"):
            if "tfidf_vectorizer" in globals(): artifact["preprocess"]["vectorizer"] = tfidf_vectorizer
            if "tfidf_svd" in globals():        artifact["preprocess"]["svd_50"] = tfidf_svd
            if "tfidf_vectorizer" in globals():
                artifact["vocab_or_tokens"]["tokens"] = safe_feature_names(tfidf_vectorizer, limit=VOCAB_LIMIT)

        elif emb_name.startswith("W2V"):
            # Save PCA50 used to build X_50 (optional but helpful)
            if pca_w2v_50 is not None:
                artifact["preprocess"]["pca_50"] = pca_w2v_50

            # Save W2V vocab (limited)
            if "w2v" in globals():
                try:
                    vocab = w2v.wv.index_to_key[:VOCAB_LIMIT]
                    artifact["vocab_or_tokens"]["tokens"] = vocab
                    # Save the actual Word2Vec model file separately (gensim)
                    w2v_path = os.path.join(ART_DIR, "word2vec.model")
                    if not os.path.exists(w2v_path):
                        w2v.save(w2v_path)
                except Exception:
                    pass

        elif emb_name.startswith("E5"):
            if pca_e5_50 is not None:
                artifact["preprocess"]["pca_50"] = pca_e5_50
            # Save model name if you used SentenceTransformer
            artifact["preprocess"]["sentence_transformer_model"] = "intfloat/multilingual-e5-base"

        # Save as joblib
        out_file = os.path.join(ART_DIR, f"{emb_name}__{algo_name}.joblib")
        joblib.dump(artifact, out_file)

        saved_index.append({"embedding": emb_name, "algo": algo_name, "file": out_file})

# Save an index json
index_path = os.path.join(ART_DIR, "INDEX.json")
with open(index_path, "w", encoding="utf-8") as f:
    json.dump(saved_index, f, indent=2)

print("Saved artifacts to:", ART_DIR)
print("Index:", index_path)
print("First 5 saved:", saved_index[:5])

In [ ]:
import pandas as pd
import os

# 1. Select the best Embedding and Model combination for the background map
# TF-IDF + KMeans is generally the most stable choice for visualization
target_emb = "TFIDF_SVD50"
target_algo = "KMeans"

# 2. Retrieve the 50-dimensional data (already normalized in your EMBEDDINGS dict)
X_for_map = EMBEDDINGS[target_emb]["X_k"] 

# 3. Use your custom project_2d function to reduce dimensions to 2D (using PCA)
# This returns a NumPy array with 2 columns representing x and y coordinates
Z_2d = project_2d(X_for_map, method="pca", seed=SEED)

# 4. Extract the cluster labels from your FINAL dictionary
labels_for_map = FINAL[target_emb][target_algo]["labels"]

# 5. Create a DataFrame and save it to the artifacts directory
df_cluster_map = pd.DataFrame({
    'x': Z_2d[:, 0],
    'y': Z_2d[:, 1],
    'cluster': labels_for_map
})

# Define the output path using your ART_DIR variable
map_output_path = os.path.join(ART_DIR, "cluster_map.csv")
df_cluster_map.to_csv(map_output_path, index=False)

print(f"Success! Cluster map data saved to: {map_output_path}")